In [ ]:
import numpy as np
from scipy.integrate import odeint
import pickle
import pyabc
from pyabc import ABCSMC, Distribution, RV, MultivariateNormalTransition, QuantileEpsilon
import tempfile

_ = np.random.seed(0)

In [ ]:
def ode_system(y, t, beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H, N):
    S, E, I, H, R, D = y
    dSdt = - (beta_I * I + beta_H * H) * S / N
    dEdt = (beta_I * I + beta_H * H) * S / N - kappa * E
    dIdt =  kappa * E - (alpha + gamma_I + delta_I) * I
    dHdt = alpha * I - (gamma_H + delta_H) * H
    dRdt = gamma_I * I + gamma_H * H
    dDdt = delta_I * I + delta_H *H
    return [dSdt, dEdt, dIdt, dHdt, dRdt, dDdt]

def simulate(parameters, ic, T):
    beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H = parameters
    S0, E0, I0, H0, R0, D0 = ic

    N = S0 + E0 + I0 + H0 + R0 + D0
    y0=ic
    
    t = np.arange(T+1)
    ret = odeint(ode_system, y0, t, args=(beta_I, beta_H, kappa, alpha, gamma_I, delta_I, gamma_H, delta_H, N))

    cases1 = kappa * ret[1:,1]
    cases2 = alpha * ret[1:,2]
    cases3 = delta_I * ret[1:,2] + delta_H * ret[1:,3]
    
    return np.stack([cases1, cases2, cases3], axis=1)
    
def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return np.asarray(with_noise)

In [ ]:
N  = 100000
E0 = 0
I0 = 10
H0 = 0
R0 = 0
D0 = 0
S0 = N - E0 - I0 - H0 - R0 - D0

duration=100

init_cond = [S0, E0, I0, H0, R0, D0]

In [ ]:
with open('../../Data/Model2/sim_dataset.pkl', 'rb') as f:
    simulation_dataset = pickle.load(f)

In [ ]:
def model(params):
    pvec = [
        params["beta_I"],
        params["beta_H"],
        params["kappa"],     
        params["alpha"],    
        params["gamma_I"],    
        params["delta_I"],
        params["gamma_H"],
        params["delta_H"]
    ]
    sim = simulate(pvec, init_cond, T=100)
    
    return {'cases1': poisson_noise(sim[:,0]),
            'cases2': poisson_noise(sim[:,1]),
            'cases3': poisson_noise(sim[:,2])}

In [ ]:
import pandas as pd
import numpy as np
import time
import pickle 

from sbi.utils import BoxUniform
from scipy.integrate import odeint
from sbi.utils import MultipleIndependent

import matplotlib.pyplot as plt

import torch
from torch.distributions import Uniform, Exponential, Cauchy

_ = torch.manual_seed(0)
_ = np.random.seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device) 

In [ ]:
import torch.nn as nn
from sbi.inference import NPE
from sbi.neural_nets import posterior_nn

In [ ]:
low  = torch.tensor([0.6, 0.6, 0.1, 0.1, 0.05, 0.001, 0.05, 0.01])
high = torch.tensor([1.5, 1.5, 0.5, 0.3, 0.4, 0.02, 0.2, 0.3])

prior = BoxUniform(low=low, high=high)

In [ ]:
class LSTMembedding(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=128, output_dim=30, num_layers=1,bidirectional=True):
        super().__init__()

        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, bidirectional=bidirectional, batch_first = True)
        self.fc = nn.Linear(hidden_dim * (2 if bidirectional else 1), output_dim)
        self.tanh = nn.Tanh()
        self.dropout = nn.Dropout(0.1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)  
        last_hidden = lstm_out[:, -1, :]
        last_hidden = self.dropout(last_hidden)
        out = self.fc(last_hidden)   
        return out

In [ ]:
embedding_net =LSTMembedding(input_dim=3, hidden_dim=128, output_dim=30).to(device)
embedding_net

In [ ]:
eps = QuantileEpsilon(initial_epsilon='from_sample', alpha=0.2)
transition = MultivariateNormalTransition(scaling=0.5)

In [ ]:
p_distance = pyabc.AdaptivePNormDistance(
    p=2,
    scale_function=pyabc.distance.std,
    all_particles_for_scale=True
)

In [ ]:
low  = np.array([0.7, 0.2, 0.1, 0.1, 0.1, 0.001, 0.1, 0.001])
high = np.array([1.3, 0.8, 0.3, 0.3, 0.3, 0.05, 0.4, 0.05])

interval=high-low

In [ ]:
prior = Distribution(
    beta_I = RV("uniform", low[0], interval[0]),
    beta_H = RV("uniform", low[1], interval[1]),
    kappa  = RV("uniform", low[2], interval[2]),
    alpha  = RV("uniform", low[3], interval[3]),
    gamma_I  = RV("uniform", low[4], interval[4]),
    delta_I  = RV("uniform", low[5], interval[5]),
    gamma_H  = RV("uniform", low[6], interval[6]),
    delta_H  = RV("uniform", low[7], interval[7])
)

In [ ]:
param_names = ["beta_I", "beta_H", "kappa", "alpha", "gamma_I", "delta_I", "gamma_H", "delta_H"]

In [ ]:
pnpe_samples=[]

for i, sim in enumerate(simulation_dataset):
    obs_data = sim['poisson']
    obs_dict={"cases1": obs_data[:,0],
             "cases2": obs_data[:,1],
             "cases3": obs_data[:,2]}
 
    db_path = "sqlite:///" + tempfile.mkstemp(suffix=f"abc_{i}.db")[1]

    abc = ABCSMC(model, prior, p_distance, population_size=1000, transitions=transition, eps=eps)
    abc.new(db_path, obs_dict)

    history = abc.run(max_total_nr_simulations=50000)

    df, weights = history.get_distribution()
    df = df[param_names]
    
    kde_estimator = pyabc.transition.MultivariateNormalTransition()
    kde_estimator.fit(df, weights)
    abc_samples = kde_estimator.rvs(50000)

    x_obs = simulation_dataset[i]['poisson']
    
    thetas = torch.tensor(np.array(abc_samples),dtype=torch.float32)

    xs = []
    for i, sample in abc_samples.iterrows():
        x_sim = poisson_noise(simulate(sample.values, init_cond, duration))
        xs.append(x_sim)
    xs = torch.tensor(xs, dtype=torch.float32)

    neural_posterior = posterior_nn(model='maf', embedding_net=embedding_net)
    inference = NPE(density_estimator=neural_posterior,device=device)
    
    density_estimator=inference.append_simulations(thetas, xs).train(training_batch_size=256)
    posterior = inference.build_posterior(density_estimator)
    samples = posterior.sample((10000,), x=x_obs)

    samples_df=pd.DataFrame(samples.cpu().numpy(),columns=param_names)
    pnpe_samples.append(samples_df)